## 6.6 卷积神经网络
### 6.6.1 LeNet
总体来看，LeNet由两个部分组成：
- 卷积编码器：有两个卷积层组成
- 全连接层密集块：由三个全连接层组成

> 输入->卷积层->汇聚层->卷积层->汇聚层->全连接层->全连接层->全连接层->输出

In [11]:
import torch
from torch import nn

net = nn.Sequential(
    nn.Conv2d(1, 6, kernel_size=5, padding=2), nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(6, 16, kernel_size=5), nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Linear(16 * 5 * 5, 120), nn.Sigmoid(),
    nn.Linear(120, 84), nn.Sigmoid(),
    nn.Linear(84, 10)
)

X = torch.rand(size=(1, 1, 28, 28), dtype=torch.float32)
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__, 'output shape: \t', X.shape)

Conv2d output shape: 	 torch.Size([1, 6, 28, 28])
Sigmoid output shape: 	 torch.Size([1, 6, 28, 28])
AvgPool2d output shape: 	 torch.Size([1, 6, 14, 14])
Conv2d output shape: 	 torch.Size([1, 16, 10, 10])
Sigmoid output shape: 	 torch.Size([1, 16, 10, 10])
AvgPool2d output shape: 	 torch.Size([1, 16, 5, 5])
Flatten output shape: 	 torch.Size([1, 400])
Linear output shape: 	 torch.Size([1, 120])
Sigmoid output shape: 	 torch.Size([1, 120])
Linear output shape: 	 torch.Size([1, 84])
Sigmoid output shape: 	 torch.Size([1, 84])
Linear output shape: 	 torch.Size([1, 10])


In [12]:
batch_size = 256

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

trans = transforms.ToTensor()

train_set = datasets.MNIST(root='./data', train=True, download=True, transform=trans)
test_set = datasets.MNIST(root='./data', train=False, download=True, transform=trans)

train_iter = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
test_iter = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)

In [13]:
def accuracy(y_hat, y):
    """计算预测正确的数量"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(dim=1)
    return (y_hat.type(y.dtype) == y).sum()

In [14]:
class Accumulator:
    def __init__(self, n):
        self.data = [0.0] * n
    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]
    def reset(self):
        self.data = [0.0] * len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

In [15]:
def evaluate_accuracy_gpu(net, data_iter, device=None):
    if isinstance(net, nn.Module):
        net.eval()
        if not device:
            device = next(iter(net.parameters())).device
            
    metric = Accumulator(2)
    with torch.no_grad():
        for X, y in data_iter:
            if isinstance(X, list):
                X = [x.to(device) for x in X]
            else:
                X = X.to(device)
            y = y.to(device)
            metric.add(accuracy(net(X), y), y.numel())
    return metric[0] / metric[1]    
        

In [16]:
import time
class Timer:
    def __init__(self):
        self.start_time = None
        self.total = 0.0
    def start(self):
        self.start_time = time.time()
    def stop(self):
        self.total += time.time() - self.start_time
    def sum(self):
        return self.total

In [17]:
import matplotlib.pyplot as plt
class Animator:
    def __init__(self, xlabel=None, ylabel=None, legend=None, xlim=None, ylim=None):
        self.xlabel = xlabel
        self.ylabel = ylabel
        self.legend = legend
        self.xlim = xlim
        self.ylim = ylim
        self.X = []
        self.Y = []

    def add(self, x, y):
        self.X.append(x)
        self.Y.append(y)
        plt.clf()
        for i in range(len(y)):
            ys = [v[i] if v[i] is not None else float('nan') for v in self.Y]
            plt.plot(self.X, ys, label=self.legend[i] if self.legend else None)
        if self.xlabel: plt.xlabel(self.xlabel)
        if self.ylabel: plt.ylabel(self.ylabel)
        if self.legend: plt.legend()
        if self.xlim: plt.xlim(self.xlim)
        if self.ylim: plt.ylim(self.ylim)
        plt.pause(0.01)

In [18]:
def train_ch6(net, train_iter, test_iter, num_epochs, lr, device):
    """用GPU训练模型（无动画版）"""
    def init_weights(m):
        if type(m) == nn.Linear or type(m) == nn.Conv2d:
            nn.init.xavier_uniform_(m.weight)
    net.apply(init_weights)
    print('training on', device)
    net.to(device)
    optimizer = torch.optim.SGD(net.parameters(), lr=lr)
    loss = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        net.train()
        metric = Accumulator(3)  # 累加 训练损失总和、正确预测数、样本数
        for X, y in train_iter:
            optimizer.zero_grad()
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            l = loss(y_hat, y)
            l.backward()
            optimizer.step()
            with torch.no_grad():
                metric.add(l * X.shape[0], accuracy(y_hat, y), X.shape[0])

        train_loss = metric[0] / metric[2]
        train_acc = metric[1] / metric[2]
        test_acc = evaluate_accuracy_gpu(net, test_iter)
        print(f'epoch {epoch+1}: loss {train_loss:.3f}, '
              f'train acc {train_acc:.3f}, test acc {test_acc:.3f}')

In [19]:
def try_gpu(i=0):
    """如果存在，则返回gpu(i)，否则返回cpu()"""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

In [20]:
lr, num_epochs = 0.9, 10
train_ch6(net, train_iter, test_iter, num_epochs, lr, try_gpu())

training on cuda:0
epoch 1: loss 2.320, train acc 0.103, test acc 0.114
epoch 2: loss 2.304, train acc 0.108, test acc 0.114
epoch 3: loss 2.302, train acc 0.109, test acc 0.114
epoch 4: loss 2.301, train acc 0.111, test acc 0.114
epoch 5: loss 2.273, train acc 0.147, test acc 0.330
epoch 6: loss 0.999, train acc 0.665, test acc 0.812
epoch 7: loss 0.392, train acc 0.878, test acc 0.911
epoch 8: loss 0.259, train acc 0.920, test acc 0.918
epoch 9: loss 0.197, train acc 0.939, test acc 0.944
epoch 10: loss 0.162, train acc 0.950, test acc 0.947
